# BrandDeck: расширенный набор и проверка новых дизайн-систем

Загрузите папку **lct_cpu_expanded** целиком в MyDrive. Исходный ноутбук сохранён. Используйте отдельную папку: старые веса и чекпоинты относятся к другому набору.

Train/validation/test разделены по семействам до перестановок кандидатов. Validation используется для выбора адаптера; test оценивается только после экспорта GGUF, решение о включении адаптера по нему не меняется. Не подбирайте настройки по test. Метрики отражают слабую разметку выбора макета, а не экспертную оценку качества дизайна. Большинство новых материалов созданы программно/с помощью ИИ; это ограничение разнообразия источников.

Обучение требует Colab GPU (A100), экспорт и итоговая проверка выполняются на CPU.


## 1. Drive и проверка файлов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
ROOT = Path('/content/drive/MyDrive/lct_cpu_expanded')
manifest = json.loads((ROOT / 'data/manifest.json').read_text())
assert manifest.get('profile') == 'cpu', 'Нужен новый комплект lct_cpu'
for relative, expected in manifest['files'].items():
    path = (ROOT / relative).resolve()
    assert path.is_relative_to(ROOT.resolve()) and path.is_file(), relative
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, relative
PROFILE = json.loads((ROOT / 'scripts/cpu_profile.json').read_text())
print('Строк:', manifest['train_rows'], 'проверка:', manifest['validation_rows'])
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import load_records, check_split
parts = {name: load_records(ROOT / 'data' / (name + '.jsonl'))
         for name in ('train', 'validation', 'test')}
check_split(parts['train'], parts['validation'], parts['test'])
assert manifest['split_unit'] == 'design_family'
for name, rows in parts.items():
    print(name, 'строк:', len(rows),
          'групп:', len({r['provenance']['group'] for r in rows}),
          'семейств:', len({r['provenance']['design_family'] for r in rows}))
print('Пересечений нет. Test не участвует в обучении и выборе адаптера.')


## 2. Окружение Python 3.12
uv устанавливает отдельный Python и pip, обходя ошибку системного `venv`.
Вывод установки отображается в ячейке. Ядро Colab перезапускать не нужно.

In [ ]:
from pathlib import Path
import os, subprocess, sys, uuid

def run_visible(command):
    # Colab does not always display inherited subprocess stdout/stderr.
    with subprocess.Popen(command, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                          errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

def ensure_python312(destination):
    destination = Path(destination)
    executable = 'Scripts/python.exe' if os.name == 'nt' else 'bin/python'
    python = destination / executable
    healthy = False
    if python.is_file():
        probe = subprocess.run([str(python), '-c',
            'import sys, pip; assert sys.version_info[:2] == (3, 12)'],
            capture_output=True, text=True)
        healthy = probe.returncode == 0
    if not healthy:
        run_visible([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
        # Preserve failed environments; never overlay another Python version.
        if destination.exists():
            destination = destination.with_name(destination.name + '-' + uuid.uuid4().hex[:8])
        run_visible([sys.executable, '-m', 'uv', 'venv', '--python', '3.12',
                     '--seed', str(destination)])
        python = destination / executable
    run_visible([str(python), '-c',
        'import sys, pip; assert sys.version_info[:2] == (3, 12); print(sys.version)'])
    return destination, str(python)


TRAIN_ENV, TRAIN_PY = ensure_python312(globals().get('TRAIN_ENV', '/content/lct-cpu-train-py312'))
run_visible([TRAIN_PY, '-m', 'pip', 'install', '-r', str(ROOT / 'scripts/requirements-a100.txt')])
os.environ['HF_HOME'] = '/content/lct-model-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
LOCAL_SCRIPTS = Path('/content/lct-cpu-scripts')
shutil.copytree(ROOT / 'scripts', LOCAL_SCRIPTS, dirs_exist_ok=True)

## 3. Обучение с полными чекпоинтами
`RESUME='auto'` продолжает прерванный запуск; завершённый не повторяет.
Для нового эксперимента используйте `'none'`, для конкретного — имя из `runs/`.
Чекпоинт после первого шага, каждых 10 шагов и каждой эпохи включает веса,
оптимизатор, scheduler, RNG и Trainer. На Drive хранятся две целые копии
с контрольными суммами. После обрыва повторите ячейки 1–3 с прежними параметрами.

In [ ]:
RESUME = 'auto'
EPOCHS = 2
MAX_LENGTH = 4096
run_visible([TRAIN_PY, '-u', str(LOCAL_SCRIPTS / 'train_layout.py'),
    '--drive-root', str(ROOT), '--model', PROFILE['base_model'],
    '--revision', PROFILE['base_revision'], '--epochs', str(EPOCHS),
    '--max-length', str(MAX_LENGTH), '--save-steps', '10',
    '--keep-checkpoints', '2', '--resume', RESUME])
latest = json.loads((ROOT / 'weights/latest.json').read_text())
metrics = json.loads((ROOT / latest['adapter_dir'] / 'metrics.json').read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 4. Сборка инструментов экспорта
Конвертер имеет отдельное окружение, поэтому его зависимости не меняют обучение.
Версия llama.cpp совпадает с Docker-образом VPS. Сборка использует только CPU.

In [ ]:
EXPORT_ENV, EXPORT_PY = ensure_python312(globals().get('EXPORT_ENV', '/content/lct-cpu-export-py312'))
run_visible(['apt-get', 'update', '-qq'])
run_visible(['apt-get', 'install', '-y', '-qq', 'git', 'cmake', 'build-essential', 'libcurl4-openssl-dev'])
LLAMA = Path('/content/lct-llama-cpu')
LLAMA.mkdir(exist_ok=True)
if not (LLAMA / '.git').exists():
    run_visible(['git', '-C', str(LLAMA), 'init'])
    run_visible(['git', '-C', str(LLAMA), 'remote', 'add', 'origin', 'https://github.com/ggml-org/llama.cpp.git'])
run_visible(['git', '-C', str(LLAMA), 'fetch', '--depth', '1', 'origin', PROFILE['llama_cpp_revision']])
run_visible(['git', '-C', str(LLAMA), 'checkout', '--detach', PROFILE['llama_cpp_revision']])
run_visible([EXPORT_PY, '-m', 'pip', 'install', '-r', str(LLAMA / 'requirements/requirements-convert_hf_to_gguf.txt'), 'psutil>=5.9'])
run_visible(['cmake', '-S', str(LLAMA), '-B', str(LLAMA / 'build'),
    '-DCMAKE_BUILD_TYPE=Release', '-DGGML_CUDA=OFF', '-DLLAMA_BUILD_TESTS=OFF'])
run_visible(['cmake', '--build', str(LLAMA / 'build'), '--config', 'Release',
    '--target', 'llama-quantize', 'llama-server', '--parallel', '2'])

## 5. GGUF, проверка на CPU и сохранение на Drive
Экспортируются база Q4_K_M и адаптер F16 GGUF. Проверяются реальная загрузка
на CPU, память, JSON-план и весь validation после квантования.
LoRA автоматически включается в приложении, только если превосходит базу
и эвристику на этой проверке и выдаёт корректные ответы. Иначе веса сохраняются,
но выбор макетов остаётся эвристическим. Это не замена визуальной проверке PPTX.
Во время экспорта Colab остаётся включённым. При сбое повторите эту ячейку;
после сброса среды восстановите 1, 2 и 4 — обучение повторять не требуется.

In [ ]:
run_visible([EXPORT_PY, '-u', str(LOCAL_SCRIPTS / 'export_cpu.py'),
    '--drive-root', str(ROOT), '--llama-root', str(LLAMA)])
exported = json.loads((ROOT / 'exports/latest.json').read_text())
test_path = ROOT / Path(exported['archive']).parent / 'cpu/test_report.json'
final_test = json.loads(test_path.read_text())
print('Новые семейства — финальный test (не использовать для подбора параметров):')
for kind in ('baseline', 'adapted'):
    print(kind, {k: v for k, v in final_test[kind].items() if k != 'predictions'})
print('Решение по validation:', final_test['frozen_enable_layout_adapter'])


## 6. Скачать готовый комплект
Архив `cpu_bundle.zip` содержит обе модели, конфигурацию и отчёт проверки.
Распакуйте в `lct_vk/models/` на VPS; получится `models/cpu/`.
Инструкция запуска — `training/CPU_README.md` в репозитории и README внутри архива.
После скачивания можно завершить сеанс Colab: VPS работает самостоятельно.

In [ ]:
from google.colab import files
exported = json.loads((ROOT / 'exports/latest.json').read_text())
archive = (ROOT / exported['archive']).resolve()
assert archive.is_relative_to(ROOT.resolve())
sha = hashlib.sha256()
with archive.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        sha.update(block)
assert sha.hexdigest() == exported['sha256'], 'Архив на Drive повреждён'
print('LoRA включён:', exported['enable_layout_adapter'])
print('Архив:', archive)
files.download(str(archive))